# Projected Quantum Kernel Features

A fidelity quantum kernel compares whole quantum states with a global overlap. That can be powerful, but it can also become uninformative when states are too far apart in Hilbert space or when measurements are noisy.

Projected quantum kernels take a different route: encode the data into a quantum state, measure local observables, and build a classical kernel from those projected features. This notebook implements that idea locally with statevector simulation.

For the hackathon story, this is a strong companion to `QSVC`: it is more hardware-friendly and gives you a way to discuss kernel concentration instead of only chasing accuracy.

## 1. Imports

This notebook reuses the same quantum-teacher image task so the projected features can be compared against classical baselines and the full fidelity kernel.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, balanced_accuracy_score, classification_report, f1_score
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVC

from qiskit.circuit.library import n_local, zz_feature_map
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_algorithms.optimizers import COBYLA, SPSA
from qiskit_machine_learning.algorithms import QSVC
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.neural_networks import SamplerQNN

SEED = 8398
np.random.seed(SEED)

## 2. Helpers

The new pieces are:

- `build_local_observables`: creates local Pauli and same-Pauli pair measurements.
- `projected_quantum_features`: encodes each sample and records those local expectation values.

In [ ]:
def evaluate_predictions(name, y_true, y_pred, *, show_confusion=True):
    """Print metrics that are useful for balanced binary classification."""
    print(f"\n{name}")
    print("balanced accuracy:", balanced_accuracy_score(y_true, y_pred))
    print("defect F1 score  :", f1_score(y_true, y_pred, zero_division=0))
    print(classification_report(y_true, y_pred, target_names=["clean", "defect"], zero_division=0))

    if show_confusion:
        ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=["clean", "defect"])
        plt.title(name)
        plt.show()


def evaluate_classifier(name, model, X_eval, y_eval, *, show_confusion=True):
    y_pred = model.predict(X_eval)
    evaluate_predictions(name, y_eval, y_pred, show_confusion=show_confusion)
    return y_pred


def balanced_subset(X_data, y_data, n_per_class=24, seed=SEED):
    """Make a small balanced set for slower quantum experiments."""
    rng = np.random.default_rng(seed)
    chosen = []
    for cls in np.unique(y_data):
        cls_idx = np.flatnonzero(y_data == cls)
        take = min(n_per_class, len(cls_idx))
        chosen.extend(rng.choice(cls_idx, size=take, replace=False).tolist())
    rng.shuffle(chosen)
    return X_data[chosen], y_data[chosen]


def latent_to_texture_images(X_latent, size=8):
    """Convert low-dimensional latent vectors into small grayscale texture images.

    The image is only a visualization/carrier for the latent features. The label is
    not created by inserting a visible blob; it is created by a quantum-kernel teacher.
    """
    grid = np.linspace(-1.0, 1.0, size)
    rr, cc = np.meshgrid(grid, grid, indexing="ij")
    images = []

    for x in X_latent:
        terms = [
            np.sin(np.pi * rr + x[0]),
            np.cos(np.pi * cc + x[1]),
            np.sin(np.pi * (rr + cc) + x[2]),
            np.cos(np.pi * (rr - cc) + x[3]),
        ]
        if len(x) > 4:
            terms.append(np.sin(np.pi * rr * cc + x[4]))
        if len(x) > 5:
            terms.append(np.cos(np.pi * (rr**2 - cc**2) + x[5]))

        texture = np.mean(terms, axis=0)
        texture = (texture - texture.min()) / (texture.max() - texture.min() + 1e-12)
        texture = 0.25 + 0.55 * texture
        images.append(texture)

    return np.array(images)


def make_quantum_teacher_dataset(
    *,
    n_samples=100,
    n_qubits=5,
    n_centers=12,
    feature_reps=1,
    image_size=8,
    seed=SEED,
):
    """Create a synthetic image dataset whose labels are generated in quantum feature space.

    A few random centers act like support vectors. The label is the sign of a weighted
    sum of quantum kernel similarities to those centers. This is an engineered task,
    not a medical claim, but it is much closer to the theory behind quantum kernels.
    """
    rng = np.random.default_rng(seed)
    X_latent = rng.uniform(0.0, 2 * np.pi, size=(n_samples, n_qubits))
    centers = rng.uniform(0.0, 2 * np.pi, size=(n_centers, n_qubits))

    feature_map = zz_feature_map(feature_dimension=n_qubits, reps=feature_reps)
    quantum_kernel = FidelityQuantumKernel(feature_map=feature_map)

    center_kernel = quantum_kernel.evaluate(x_vec=X_latent, y_vec=centers)
    weights = rng.normal(size=n_centers)
    raw_scores = center_kernel @ weights
    threshold = np.median(raw_scores)
    labels = (raw_scores > threshold).astype(int)

    images = latent_to_texture_images(X_latent, size=image_size)

    return {
        "X_latent": X_latent,
        "images": images,
        "labels": labels,
        "centers": centers,
        "weights": weights,
        "scores": raw_scores,
        "threshold": threshold,
        "feature_map": feature_map,
        "quantum_kernel": quantum_kernel,
    }


def show_labeled_textures(images, labels, scores=None, n_show=12, cols=6):
    order = np.arange(min(n_show, len(images)))
    rows = int(np.ceil(len(order) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.5, rows * 1.7))
    axes = np.ravel(axes)

    for ax, idx in zip(axes, order):
        ax.imshow(images[idx], cmap="gray_r", vmin=0, vmax=1)
        label = "DEFECT" if labels[idx] else "clean"
        suffix = "" if scores is None else f"\nscore={scores[idx]:.3f}"
        ax.set_title(f"#{idx} {label}{suffix}", fontsize=8, color="crimson" if labels[idx] else "black")
        ax.set_xticks([])
        ax.set_yticks([])

    for ax in axes[len(order):]:
        ax.axis("off")

    fig.tight_layout()
    plt.show()


def centered_kernel_alignment(K, y):
    """Centered alignment between a kernel matrix and the ideal label kernel y y^T."""
    y_signed = 2 * np.asarray(y) - 1
    Y = np.outer(y_signed, y_signed)
    n = K.shape[0]
    H = np.eye(n) - np.ones((n, n)) / n
    Kc = H @ K @ H
    Yc = H @ Y @ H
    denom = np.linalg.norm(Kc, "fro") * np.linalg.norm(Yc, "fro")
    return float(np.sum(Kc * Yc) / denom) if denom else np.nan



def build_local_observables(n_qubits, include_pairs=True):
    """Local Pauli and optional same-Pauli pair observables for projected features.

    `ZZFeatureMap` stores much of its information in phases. Measuring only Z can
    miss that structure, so we include X, Y, and Z local views.
    """
    observables = []
    names = []
    single_paulis = ["X", "Y", "Z"]

    for pauli in single_paulis:
        for i in range(n_qubits):
            observables.append(SparsePauliOp.from_sparse_list([(pauli, [i], 1.0)], num_qubits=n_qubits))
            names.append(f"{pauli}{i}")

    if include_pairs:
        for pauli in single_paulis:
            pair_label = pauli * 2
            for i in range(n_qubits):
                for j in range(i + 1, n_qubits):
                    observables.append(SparsePauliOp.from_sparse_list([(pair_label, [i, j], 1.0)], num_qubits=n_qubits))
                    names.append(f"{pauli}{i}{pauli}{j}")

    return observables, names


def projected_quantum_features(X_data, feature_map, observables):
    """Encode each sample, measure local observables, and return classical features."""
    params = list(feature_map.parameters)
    rows = []

    for x in X_data:
        bound = feature_map.assign_parameters(dict(zip(params, x)), inplace=False)
        state = Statevector.from_instruction(bound)
        rows.append([float(np.real(state.expectation_value(obs))) for obs in observables])

    return np.array(rows)

## 3. Generate the Same Quantum-Tailored Dataset

The defaults match the quantum-kernel notebook, including the fixed `DATA_SEED`. Keep them small until the workflow is stable.

In [ ]:
N_SAMPLES = 100
N_QUBITS = 5
N_CENTERS = 12
FEATURE_REPS = 1
IMAGE_SIZE = 8
DATA_SEED = SEED + 501

data = make_quantum_teacher_dataset(
    n_samples=N_SAMPLES,
    n_qubits=N_QUBITS,
    n_centers=N_CENTERS,
    feature_reps=FEATURE_REPS,
    image_size=IMAGE_SIZE,
    seed=DATA_SEED,
)

X_latent = data["X_latent"]
images = data["images"]
y = data["labels"]
feature_map = data["feature_map"]
quantum_kernel = data["quantum_kernel"]

show_labeled_textures(images, y, scores=data["scores"], n_show=12, cols=6)

## 4. Split Data

Projected features are computed after the split to avoid mixing train/test processing decisions.

In [ ]:
X_latent_train, X_latent_test, y_train, y_test = train_test_split(
    X_latent,
    y,
    test_size=0.30,
    random_state=SEED,
    stratify=y,
)

print("train:", X_latent_train.shape, "clean:", int((y_train == 0).sum()), "defect:", int((y_train == 1).sum()))
print("test :", X_latent_test.shape, "clean:", int((y_test == 0).sum()), "defect:", int((y_test == 1).sum()))

## 5. Classical Baselines on Latent Features

These set the floor for the projected quantum features.

In [ ]:
baseline_models = {
    "Latent Logistic Regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)),
    ]),
    "Latent RBF SVM": Pipeline([
        ("scale", StandardScaler()),
        ("model", SVC(kernel="rbf", class_weight="balanced", random_state=SEED)),
    ]),
    "Latent Random Forest": RandomForestClassifier(
        n_estimators=250,
        class_weight="balanced",
        random_state=SEED,
    ),
}

for name, model in baseline_models.items():
    model.fit(X_latent_train, y_train)
    evaluate_classifier(name, model, X_latent_test, y_test, show_confusion=False)

## 6. Build Projected Quantum Features

For five qubits, local `X_i`, `Y_i`, `Z_i` plus same-Pauli pairs gives 45 projected features. That is still small, interpretable, and much cheaper than a full kernel matrix.

In [ ]:
observables, observable_names = build_local_observables(N_QUBITS, include_pairs=True)
print("number of projected observables:", len(observables))
print("first observables:", observable_names[:8])

X_proj_train = projected_quantum_features(X_latent_train, feature_map, observables)
X_proj_test = projected_quantum_features(X_latent_test, feature_map, observables)

print("projected train shape:", X_proj_train.shape)
print("projected test shape :", X_proj_test.shape)

## 7. Train Classical Models on Projected Quantum Features

This is the projected-kernel idea in a simple form: quantum circuit first, local measurements second, classical learner third.

In [ ]:
projected_models = {
    "Projected Logistic Regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)),
    ]),
    "Projected Linear SVM": Pipeline([
        ("scale", StandardScaler()),
        ("model", SVC(kernel="linear", class_weight="balanced", random_state=SEED)),
    ]),
    "Projected RBF SVM": Pipeline([
        ("scale", StandardScaler()),
        ("model", SVC(kernel="rbf", class_weight="balanced", random_state=SEED)),
    ]),
    "Projected Random Forest": RandomForestClassifier(
        n_estimators=250,
        class_weight="balanced",
        random_state=SEED,
    ),
}

for name, model in projected_models.items():
    model.fit(X_proj_train, y_train)
    evaluate_classifier(name, model, X_proj_test, y_test, show_confusion=False)

## 8. Alignment: Full Fidelity Kernel vs Projected Feature Kernel

This does not measure speedup. It measures whether the geometry created by a kernel agrees with the labels. If the full fidelity kernel has very low off-diagonal variance, it may be concentrating.

In [ ]:
X_align, y_align = balanced_subset(X_latent_train, y_train, n_per_class=24)
X_align_proj = projected_quantum_features(X_align, feature_map, observables)

K_fidelity = quantum_kernel.evaluate(x_vec=X_align)
K_projected_linear = X_align_proj @ X_align_proj.T
X_align_proj_scaled = StandardScaler().fit_transform(X_align_proj)
X_align_scaled = StandardScaler().fit_transform(X_align)
K_projected_rbf = rbf_kernel(X_align_proj_scaled, gamma=1 / X_align_proj_scaled.shape[1])
K_classical_rbf = rbf_kernel(X_align_scaled, gamma=1 / X_align_scaled.shape[1])

for name, K in [
    ("fidelity quantum kernel", K_fidelity),
    ("projected linear kernel", K_projected_linear),
    ("projected RBF kernel", K_projected_rbf),
    ("classical latent RBF", K_classical_rbf),
]:
    off_diag = K[~np.eye(len(K), dtype=bool)]
    print(f"{name:24s} alignment={centered_kernel_alignment(K, y_align): .4f} offdiag_var={np.var(off_diag): .6f}")

## 9. How to Use This Direction

Projected quantum features are a good second pitch after `QSVC`:

- They are easier to run on hardware because they use local measurements.
- They avoid relying entirely on global state overlaps.
- They create classical features you can inspect, rank, or feed into many models.
- They are not automatically better; the alignment cell tells you whether the projection preserved label-relevant structure.